In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q slither-analyzer solc-select

import os, re, json, time, subprocess, requests
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive/rugpull_research'
KEY  = 'ISI_API_KEY_ETHERSCAN_ANDA'
RAW  = f'{BASE}/raw_etherscan'
SRC  = f'{BASE}/src_v2'
os.makedirs(RAW, exist_ok=True); os.makedirs(SRC, exist_ok=True)

rows = []
for cls, label in [('rugpull', 1), ('legitimate', 0)]:
    folder = f'{BASE}/contracts/{cls}'
    for f in os.listdir(folder):
        if f.endswith('.sol'):
            rows.append((f[:-4].lower(), label, f'{folder}/{f}'))
meta = pd.DataFrame(rows, columns=['address', 'label', 'local_path']).drop_duplicates('address')
print(meta['label'].value_counts())

In [ ]:
def fetch(addr):
    p = f'{RAW}/{addr}.json'
    if os.path.exists(p):
        return json.load(open(p))
    for _ in range(4):
        try:
            r = requests.get('https://api.etherscan.io/v2/api', timeout=30, params=dict(
                chainid=1, module='contract', action='getsourcecode', address=addr, apikey=KEY)).json()
            if isinstance(r.get('result'), list) and r['result']:
                res = r['result'][0]
                json.dump(res, open(p, 'w'))
                time.sleep(0.22)
                return res
        except Exception:
            pass
        time.sleep(1.5)
    return None

for i, a in enumerate(meta['address']):
    res = fetch(a)
    if res and res.get('Proxy') == '1' and res.get('Implementation'):
        fetch(res['Implementation'].lower())
    if i % 200 == 0:
        print(i, '/', len(meta))
print('selesai:', len(os.listdir(RAW)), 'file')

In [ ]:
FEAT = json.load(open(f'{BASE}/features/feature_cols.json'))
OUT  = f'{BASE}/features/feature_v2.csv'
SL_COLS = ['total_vulnerabilities','high_severity','medium_severity','low_severity','informational','optimization',
           'has_assembly','has_low_level_calls','has_dead_code','has_shadowing','has_costly_loop','has_reentrancy']

def unpack(res):
    s = (res.get('SourceCode') or '').strip()
    if not s: return None
    try:
        if s.startswith('{{'):
            j = json.loads(s[1:-1]); files = {p: v['content'] for p, v in j['sources'].items()}
            remaps = j.get('settings', {}).get('remappings', [])
        elif s.startswith('{'):
            j = json.loads(s)
            if 'sources' in j:
                files = {p: v['content'] for p, v in j['sources'].items()}; remaps = j.get('settings', {}).get('remappings', [])
            else:
                files = {p: v['content'] for p, v in j.items()}; remaps = []
        else:
            files = {f"{res.get('ContractName') or 'Contract'}.sol": s}; remaps = []
    except Exception:
        return None
    return files, remaps

def write_tree(addr, files):
    d = f'{SRC}/{addr}'
    for p, c in files.items():
        fp = os.path.normpath(os.path.join(d, p.lstrip('/')))
        if not fp.startswith(d): continue
        os.makedirs(os.path.dirname(fp), exist_ok=True)
        with open(fp, 'w', encoding='utf-8') as fh: fh.write(c)
    return d

def pick_main(files, name):
    if name:
        pat = re.compile(rf'\b(contract|library|interface)\s+{re.escape(name)}\b')
        for p, c in files.items():
            if pat.search(c): return p.lstrip('/')
    return max(files, key=lambda p: len(files[p])).lstrip('/')

def solc_ver(res):
    m = re.search(r'(\d+\.\d+\.\d+)', res.get('CompilerVersion', '') or '')
    return m.group(1) if m else None

installed = set()
def run_slither(d, main, ver, remaps):
    if not ver: return None, 'no_version'
    if ver not in installed:
        subprocess.run(['solc-select', 'install', ver], capture_output=True); installed.add(ver)
    env = {**os.environ, 'SOLC_VERSION': ver}
    cmd = ['slither', main, '--json', '-'] + (['--solc-remaps', ' '.join(remaps)] if remaps else [])
    try:
        r = subprocess.run(cmd, cwd=d, env=env, capture_output=True, text=True, timeout=180)
        data = json.loads(r.stdout) if r.stdout.strip() else None
        if data and data.get('success') is not False and 'results' in data:
            return data['results'].get('detectors', []), 'ok'
        return None, 'slither_error'
    except subprocess.TimeoutExpired:
        return None, 'timeout'
    except Exception:
        return None, 'exception'

def slither_feats(det):
    chk = [x['check'] for x in det]
    n = lambda k: sum(1 for x in det if x.get('impact') == k)
    return dict(total_vulnerabilities=len(det), high_severity=n('High'), medium_severity=n('Medium'),
                low_severity=n('Low'), informational=n('Informational'), optimization=n('Optimization'),
                has_assembly=int('assembly' in chk), has_low_level_calls=int('low-level-calls' in chk),
                has_dead_code=int('dead-code' in chk), has_shadowing=int('shadowing-local' in chk),
                has_costly_loop=int('costly-loop' in chk), has_reentrancy=int(any('reentrancy' in c for c in chk)))

def text_feats(source):
    s = source.lower()
    rx = lambda pat: int(bool(re.search(pat, s)))
    return dict(
        has_mint=rx(r'\bmint\b|\b_mint\b|\bminttoken\b'), has_blacklist=rx(r'\bblacklist\b|\bblocklist\b|\bblocked\b'),
        has_pause=rx(r'\bpause\b|\bpaused\b|\bunpause\b'), has_selfdestruct=rx(r'\bselfdestruct\b|\bsuicide\b'),
        has_owner_only=rx(r'\bonlyowner\b|\brequire.*owner\b'), has_fee_modifier=rx(r'\btaxfee\b|\bliquidityfee\b|\b_taxfee\b|\bsetfee\b'),
        has_max_tx=rx(r'\bmaxtxamount\b|\b_maxtx\b|\bmaxwallet\b'), has_whitelist=rx(r'\bwhitelist\b|\bexcluded\b|\b_isexcluded\b'),
        has_transfer_lock=rx(r'\blocktime\b|\btimelock\b|\blocked\b'), has_proxy=rx(r'\bdelegatecall\b|\bproxy\b|\bupgradeable\b'),
        has_renounce=rx(r'\brenounceownership\b'), has_burn=rx(r'\bburn\b|\b_burn\b|\bburntoken\b'),
        func_count=len(re.findall(r'\bfunction\b', s)), modifier_count=len(re.findall(r'\bmodifier\b', s)),
        event_count=len(re.findall(r'\bevent\b', s)), require_count=len(re.findall(r'\brequire\b', s)),
        code_length=len(s))

In [ ]:
done = set(pd.read_csv(OUT)['address']) if os.path.exists(OUT) else set()
for i, r in meta.iterrows():
    a = r['address']
    if a in done: continue
    res = json.load(open(f'{RAW}/{a}.json')) if os.path.exists(f'{RAW}/{a}.json') else None
    is_proxy = 0
    if res and res.get('Proxy') == '1' and res.get('Implementation'):
        ip = f"{RAW}/{res['Implementation'].lower()}.json"
        if os.path.exists(ip):
            r2 = json.load(open(ip))
            if r2.get('SourceCode'): res, is_proxy = r2, 1
    u = unpack(res) if res else None
    if u:
        files, remaps = u
        kind = 'multi_file' if len(files) > 1 else 'single'
        source = '\n'.join(files.values())
        d = write_tree(a, files)
        det, status = run_slither(d, pick_main(files, res.get('ContractName')), solc_ver(res), remaps)
        n_files = len(files)
    else:
        source = open(r['local_path'], encoding='utf-8', errors='ignore').read()
        kind, det, status, n_files = 'local_fallback', None, 'no_verified_source', 1
    rec = dict(address=a, label=r['label'])
    rec.update(slither_feats(det) if det is not None else {c: np.nan for c in SL_COLS})
    rec.update(text_feats(source))
    rec.update(slither_status=status, source_kind=kind, is_proxy=is_proxy, n_files=n_files)
    pd.DataFrame([rec]).to_csv(OUT, mode='a', header=not os.path.exists(OUT), index=False)
    if i % 100 == 0: print(i, '/', len(meta))

v2 = pd.read_csv(OUT)
print(len(v2), 'baris')
print('\nStatus Slither per kelas:');  print(pd.crosstab(v2['label'], v2['slither_status'], normalize='index').round(3))
print('\nJenis sumber per kelas:');    print(pd.crosstab(v2['label'], v2['source_kind']))
print('\nProxy per kelas:');           print(pd.crosstab(v2['label'], v2['is_proxy']))

Langkah 1: cek 46 kontrak yang bermasalah

In [ ]:
v2 = pd.read_csv(f'{BASE}/features/feature_v2.csv')
print(pd.crosstab(v2['label'], v2['slither_status']))

for a in v2[v2['slither_status']=='no_verified_source']['address'].head(8):
    p = f'{RAW}/{a}.json'
    if not os.path.exists(p):
        print(a, '-> respons Etherscan TIDAK ADA (fetch gagal)'); continue
    res = json.load(open(p)); s = res.get('SourceCode') or ''
    print(a, '| panjang SourceCode:', len(s), '| awal:', repr(s[:50]), '| ABI:', (res.get('ABI') or '')[:35])

# bukti ketimpangan lama sudah hilang?
ok = v2[v2['slither_status']=='ok']
sl = ['total_vulnerabilities','high_severity','medium_severity','low_severity','informational','optimization']
print(pd.crosstab(ok['label'], ok[sl].sum(axis=1)==0, normalize='index').round(3))

hitung ulang model di data baru

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (matthews_corrcoef, roc_auc_score, average_precision_score,
                             balanced_accuracy_score, confusion_matrix)
import xgboost as xgb

main = v2[v2['slither_status']=='ok'].reset_index(drop=True)
print(main['label'].value_counts())

def make_models(seed):
    return {'LogReg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight='balanced', random_state=seed)),
            'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=seed, n_jobs=-1),
            'XGBoost': xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8,
                                         colsample_bytree=0.8, eval_metric='logloss', random_state=seed)}

def cv_eval(d, cols, n_repeats=5, n_splits=5, only=None):
    d = d.reset_index(drop=True); X = d[cols]; y = d['label'].values
    key = [c for c in FEAT if c != 'code_length']
    g = (d[key].astype(int).astype(str).agg('|'.join, axis=1) + '|' + (d['code_length']//1000).astype(int).astype(str)).factorize()[0]
    rows = []
    for rep in range(n_repeats):
        for tr, te in StratifiedGroupKFold(n_splits, shuffle=True, random_state=rep).split(X, y, g):
            if len(set(y[te])) < 2: continue
            for name, m in make_models(rep).items():
                if only and name != only: continue
                if name == 'XGBoost': m.fit(X.iloc[tr], y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
                else: m.fit(X.iloc[tr], y[tr])
                p = m.predict_proba(X.iloc[te])[:,1]; pr = (p >= 0.5).astype(int)
                tn, fp, fn, tp = confusion_matrix(y[te], pr, labels=[0,1]).ravel()
                rows.append(dict(model=name, mcc=matthews_corrcoef(y[te], pr), bal_acc=balanced_accuracy_score(y[te], pr),
                                 spec=tn/max(tn+fp,1), rec=tp/max(tp+fn,1), pr_auc_legit=average_precision_score(1-y[te], 1-p),
                                 roc_auc=roc_auc_score(y[te], p), tn=tn, fp=fp, fn=fn, tp=tp, n_legit=(y[te]==0).sum()))
    return pd.DataFrame(rows)

def report(res, n_repeats=5, name=''):
    print('\n==', name)
    print(res.groupby('model')[['mcc','bal_acc','spec','rec','pr_auc_legit','roc_auc']].agg(['mean','std']).round(3))
    print((res.groupby('model')[['tn','fp','fn','tp']].sum()/n_repeats).round(1))
    print('legit per fold:', round(res['n_legit'].mean(), 1))

report(cv_eval(main, FEAT), name='A. Semua kontrak (Slither ok)')
single = main[main['source_kind']=='single']
print(single['label'].value_counts())
report(cv_eval(single, FEAT), name='B. Hanya single-file (kontrol pembaur multi-file)')

Ablation

In [ ]:
sl_sev=['total_vulnerabilities','high_severity','medium_severity','low_severity','informational','optimization']
sl_flag=['has_assembly','has_low_level_calls','has_dead_code','has_shadowing','has_costly_loop','has_reentrancy']
rx=['has_mint','has_blacklist','has_pause','has_selfdestruct','has_owner_only','has_fee_modifier',
    'has_max_tx','has_whitelist','has_transfer_lock','has_proxy','has_renounce','has_burn']
st=['func_count','modifier_count','event_count','require_count','code_length']
assert set(sl_sev+sl_flag+rx+st)==set(FEAT)
for name, cols in {'Slither saja':sl_sev+sl_flag,'Regex saja':rx,'Struktural saja':st,
                   'Slither+Regex (tanpa ukuran)':sl_sev+sl_flag+rx,'Semua 29':FEAT}.items():
    r = cv_eval(main, cols, n_repeats=3, only='XGBoost')
    print(f"{name:32s}", r[['mcc','bal_acc','spec','pr_auc_legit']].mean().round(3).to_dict())

In [ ]:
from scipy import stats
from sklearn.metrics import roc_curve
from math import sqrt

main.to_csv(f'{BASE}/features/main_v2_ok.csv', index=False)   # simpan data analisis utama

def wilson(k, n, z=1.96):
    p = k/n; den = 1 + z*z/n; c = (p + z*z/(2*n))/den; h = z*sqrt(p*(1-p)/n + z*z/(4*n*n))/den
    return round(c-h, 3), round(c+h, 3)

def cv_eval2(d, cols, n_repeats=5, n_splits=5, only=None):
    d = d.reset_index(drop=True); X = d[cols]; y = d['label'].values
    key = [c for c in FEAT if c != 'code_length']
    g = (d[key].astype(int).astype(str).agg('|'.join, axis=1) + '|' + (d['code_length']//1000).astype(int).astype(str)).factorize()[0]
    folds, oof = [], []
    for rep in range(n_repeats):
        for k, (tr, te) in enumerate(StratifiedGroupKFold(n_splits, shuffle=True, random_state=rep).split(X, y, g)):
            if len(set(y[te])) < 2: continue
            for name, m in make_models(rep).items():
                if only and name != only: continue
                if name == 'XGBoost': m.fit(X.iloc[tr], y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
                else: m.fit(X.iloc[tr], y[tr])
                p = m.predict_proba(X.iloc[te])[:, 1]; pr = (p >= 0.5).astype(int)
                folds.append(dict(model=name, rep=rep, fold=k, mcc=matthews_corrcoef(y[te], pr),
                                  pr_auc_legit=average_precision_score(1-y[te], 1-p), roc_auc=roc_auc_score(y[te], p)))
                oof += [dict(model=name, rep=rep, i=i, y=yy, p=pp) for i, yy, pp in zip(te, y[te], p)]
    return pd.DataFrame(folds), pd.DataFrame(oof)

def paired_nb(fa, fb, metric, n_splits=5):
    """Uji t berpasangan terkoreksi (Nadeau-Bengio) untuk CV berulang. Selisih > 0 berarti fa lebih baik."""
    a = fa.sort_values(['rep','fold'])[metric].values; b = fb.sort_values(['rep','fold'])[metric].values
    d = a - b; n = len(d)
    t = d.mean() / sqrt((1/n + 1/(n_splits-1)) * d.var(ddof=1))
    return round(d.mean(), 3), round(2*(1 - stats.t.cdf(abs(t), df=n-1)), 4)

# (a) antar model
fx, ox = cv_eval2(main, FEAT)
for other in ['RandomForest', 'LogReg']:
    for met in ['mcc', 'pr_auc_legit', 'roc_auc']:
        print('XGBoost vs', other, met, '-> (selisih, p):', paired_nb(fx[fx.model=='XGBoost'], fx[fx.model==other], met))

# (b) ablation 'tanpa tiap grup' + uji terhadap model 29 fitur
sl_sev=['total_vulnerabilities','high_severity','medium_severity','low_severity','informational','optimization']
sl_flag=['has_assembly','has_low_level_calls','has_dead_code','has_shadowing','has_costly_loop','has_reentrancy']
rx=['has_mint','has_blacklist','has_pause','has_selfdestruct','has_owner_only','has_fee_modifier',
    'has_max_tx','has_whitelist','has_transfer_lock','has_proxy','has_renounce','has_burn']
st=['func_count','modifier_count','event_count','require_count','code_length']
base_f = fx[fx.model=='XGBoost']
print('\nSemua 29: MCC=%.3f' % base_f.mcc.mean())
for name, cols in {'tanpa Slither': rx+st, 'tanpa Regex': sl_sev+sl_flag+st, 'tanpa Struktural': sl_sev+sl_flag+rx,
                   'Slither saja': sl_sev+sl_flag, 'Struktural saja': st}.items():
    f_, _ = cv_eval2(main, cols, only='XGBoost')
    print(f"{name:18s} MCC={f_.mcc.mean():.3f}  PR-AUC legit={f_.pr_auc_legit.mean():.3f} | Semua29 vs ini (selisih MCC, p): {paired_nb(base_f, f_, 'mcc')}")

# (c) titik operasi: spesifisitas pada recall rug pull tertentu (deskriptif, BUKAN untuk klaim akhir)
q = ox[ox.model=='XGBoost'].groupby('i').agg(y=('y','first'), p=('p','mean')).reset_index()
fpr, tpr, thr = roc_curve(q['y'], q['p'])
for target in [0.90, 0.95, 0.99]:
    j = np.argmax(tpr >= target)
    print(f'recall rug pull >= {target:.2f} -> spesifisitas legit = {1-fpr[j]:.3f}  (threshold {thr[j]:.3f})')

# (d) CI spesifisitas dari hitungan absolut (contoh: XGBoost 72 dari 101 benar)
print('Wilson CI spesifisitas 72/101:', wilson(72, 101))

In [ ]:
os.makedirs(f'{BASE}/outputs/metrics', exist_ok=True)
m_ = main.reset_index(drop=True)
q = ox[ox.model=='XGBoost'].groupby('i').agg(y=('y','first'), p=('p','mean')).reset_index()
err = m_.loc[q['i'], ['address','label','source_kind','is_proxy','n_files','code_length','func_count']].reset_index(drop=True)
err['p_rug'] = q['p'].values
err['pred'] = (err['p_rug'] >= 0.5).astype(int)
err.to_csv(f'{BASE}/outputs/metrics/oof_xgb_v2.csv', index=False)

legit = err[err.label==0]; rug = err[err.label==1]
print('Legit salah ditandai rug pull, per jenis sumber:'); print(legit.groupby('source_kind')['pred'].agg(['mean','count']).round(3))
print('\nLegit salah ditandai, per status proxy:');         print(legit.groupby('is_proxy')['pred'].agg(['mean','count']).round(3))
print('\nRug pull terlewat (proporsi), per jenis sumber:'); print(rug.groupby('source_kind')['pred'].agg(lambda s: (s==0).mean()).round(3))
fn = rug[rug.pred==0].sort_values('p_rug')
print('\nRug pull terlewat:', len(fn), 'dari', len(rug))
print(fn[['address','source_kind','is_proxy','code_length','func_count','p_rug']].head(15).to_string(index=False))
print('\nMedian code_length:'); print(pd.Series({'rug terlewat': fn.code_length.median(),
      'rug terdeteksi': rug[rug.pred==1].code_length.median(), 'legit': legit.code_length.median()}))

In [ ]:
import shap, matplotlib.pyplot as plt
os.makedirs(f'{BASE}/outputs/shap_plots', exist_ok=True)
d = main.reset_index(drop=True); X = d[FEAT]; y = d['label'].values
key = [c for c in FEAT if c != 'code_length']
g = (d[key].astype(int).astype(str).agg('|'.join, axis=1) + '|' + (d['code_length']//1000).astype(int).astype(str)).factorize()[0]

sv = np.zeros((len(d), len(FEAT))); base = np.zeros(len(d))
for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=0).split(X, y, g):
    m = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8,
                          colsample_bytree=0.8, eval_metric='logloss', random_state=0)
    m.fit(X.iloc[tr], y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
    e = shap.TreeExplainer(m)(X.iloc[te]); sv[te] = e.values; base[te] = e.base_values

imp = pd.DataFrame({'feature': FEAT, 'mean_abs_shap': np.abs(sv).mean(0),
      'arah_korelasi': [np.corrcoef(X[f], sv[:, i])[0, 1] if X[f].nunique() > 1 else np.nan for i, f in enumerate(FEAT)]
      }).sort_values('mean_abs_shap', ascending=False)
print(imp.head(15).round(3).to_string(index=False))
imp.to_csv(f'{BASE}/outputs/shap_plots/shap_importance_v2.csv', index=False)

plt.figure(); shap.summary_plot(sv, X, show=False, max_display=15)
plt.savefig(f'{BASE}/outputs/shap_plots/shap_summary_v2.png', dpi=300, bbox_inches='tight'); plt.close()

def wf(i, name):
    ex = shap.Explanation(values=sv[i], base_values=base[i], data=X.iloc[i].values, feature_names=FEAT)
    plt.figure(); shap.plots.waterfall(ex, show=False)
    plt.savefig(f'{BASE}/outputs/shap_plots/{name}.png', dpi=300, bbox_inches='tight'); plt.close()

i_rug = err[(err.label==1)&(err.pred==1)].sample(1, random_state=1).index[0]   # dipilih acak, bukan dipilih-pilih
i_leg = err[(err.label==0)&(err.pred==0)].sample(1, random_state=1).index[0]
wf(i_rug, 'shap_waterfall_rugpull_v2'); wf(i_leg, 'shap_waterfall_legit_v2')

In [ ]:
import requests, time
# 1) CoinGecko: koin + alamat kontrak Ethereum
r = requests.get('https://api.coingecko.com/api/v3/coins/list', params={'include_platform':'true'}, timeout=60)
print('status:', r.status_code)
if r.status_code == 200:
    eth = [(c['id'], c['symbol'], c['platforms']['ethereum']) for c in r.json() if (c.get('platforms') or {}).get('ethereum')]
    print('koin dengan alamat Ethereum:', len(eth)); print(eth[:5])
else:
    print(r.text[:300])

# 2) Etherscan: transfer pertama dan terakhir sebuah token (uji pada 2 alamat legit dari data lu)
def transfer_span(addr):
    out = {}
    for sort in ['asc', 'desc']:
        j = requests.get('https://api.etherscan.io/v2/api', timeout=30, params=dict(chainid=1, module='account',
              action='tokentx', contractaddress=addr, page=1, offset=1, sort=sort, apikey=KEY)).json()
        res = j.get('result')
        out[sort] = int(res[0]['timeStamp']) if isinstance(res, list) and res else j.get('result')
        time.sleep(0.22)
    return out
for a in list(main[main.label==0]['address'].head(2)): print(a, transfer_span(a))

In [ ]:
d = main.reset_index(drop=True)
key = [c for c in FEAT if c != 'code_length']
sig = d[key].astype(int).astype(str).agg('|'.join, axis=1) + '|' + (d['code_length']//1000).astype(int).astype(str)
dd = d.loc[~sig.duplicated()].reset_index(drop=True)      # satu wakil per kelompok klon
print(dd['label'].value_counts())
report(cv_eval(dd, FEAT), name='C. Satu kontrak per kelompok klon')

In [ ]:
def eff_res(a):
    res = json.load(open(f'{RAW}/{a}.json'))
    if res.get('Proxy') == '1' and res.get('Implementation'):
        ip = f"{RAW}/{res['Implementation'].lower()}.json"
        if os.path.exists(ip):
            r2 = json.load(open(ip))
            if r2.get('SourceCode'): return r2
    return res

def solc_minor(res):
    m = re.search(r'0\.(\d+)\.\d+', res.get('CompilerVersion', '') or '')
    return int(m.group(1)) if m else np.nan

R_ = [eff_res(a) for a in main['address']]
main['solc_minor'] = [solc_minor(r) for r in R_]
print(pd.crosstab(main['label'], main['solc_minor'], normalize='index').round(3))

In [ ]:
SPAN = f'{BASE}/features/first_transfer.csv'
done = set(pd.read_csv(SPAN)['address']) if os.path.exists(SPAN) else set()
for i, a in enumerate(main['address']):
    if a in done: continue
    j = requests.get('https://api.etherscan.io/v2/api', timeout=30, params=dict(chainid=1, module='account',
          action='tokentx', contractaddress=a, page=1, offset=1, sort='asc', apikey=KEY)).json()
    res = j.get('result'); ts = int(res[0]['timeStamp']) if isinstance(res, list) and res else np.nan
    pd.DataFrame([dict(address=a, first_ts=ts)]).to_csv(SPAN, mode='a', header=not os.path.exists(SPAN), index=False)
    time.sleep(0.22)
    if i % 300 == 0: print(i, '/', len(main))

ft = pd.read_csv(SPAN).drop_duplicates('address').merge(main[['address','label']], on='address')
ft['year'] = pd.to_datetime(ft['first_ts'], unit='s').dt.year
print(pd.crosstab(ft['label'], ft['year']))
print('tanpa transfer terdeteksi:', ft['first_ts'].isna().groupby(ft['label']).sum().to_dict())

In [ ]:
# 1) ABI: apakah kontrak adalah token ERC-20? (pakai R_ dari Cek A; jangan dijalankan setelah main diubah urutannya)
def abi_names(res):
    try: return {x.get('name') for x in json.loads(res.get('ABI') or '[]') if x.get('type') == 'function'}
    except Exception: return set()
E20 = {'totalSupply','balanceOf','transfer','approve','allowance','transferFrom'}
abi_map = {a: int(E20 <= abi_names(r)) for a, r in zip(main['address'], R_)}
main['erc20_like'] = main['address'].map(abi_map)

ft = pd.read_csv(f'{BASE}/features/first_transfer.csv').drop_duplicates('address')
main = main.drop(columns=[c for c in ['first_ts','year'] if c in main.columns]).merge(ft, on='address', how='left')
main['year'] = pd.to_datetime(main['first_ts'], unit='s').dt.year
print(pd.crosstab(main['label'], main['erc20_like']))
print('\n288 tanpa transfer, apakah token ERC-20?')
print(pd.crosstab([main['label'], main['first_ts'].isna()], main['erc20_like']))

# 2) Seberapa jauh label bisa diprediksi dari ERA saja (model, bukan aturan)?
d_era = main.dropna(subset=['year', 'solc_minor']).copy()
report(cv_eval(d_era, ['solc_minor', 'year'], only='XGBoost'), name='Era saja (solc + tahun)')

# 3) Apakah fitur teratas SHAP berkorelasi dengan era kompiler?
print(main[['optimization','informational','modifier_count','func_count','code_length','solc_minor']]
      .corr(method='spearman')['solc_minor'].round(2))

# 4) Kinerja di area yang tumpang-tindih: ERC-20, peluncuran 2020-2023
sub = main[(main.erc20_like == 1) & main['year'].between(2020, 2023)]
print(sub['label'].value_counts())
if (sub.label == 0).sum() >= 30: report(cv_eval(sub, FEAT), name='ERC-20 & 2020-2023')

In [ ]:
cg = requests.get('https://api.coingecko.com/api/v3/coins/list', params={'include_platform':'true'}, timeout=60).json()
cand = pd.DataFrame([(c['id'], c['symbol'], c['platforms']['ethereum'].lower()) for c in cg
                     if (c.get('platforms') or {}).get('ethereum')], columns=['cg_id','symbol','address']).drop_duplicates('address')
cand = cand[cand['address'].str.match(r'^0x[0-9a-f]{40}$')]
dx = pd.read_csv(f'{BASE}/dataset/dianxiang/rugpull_full_dataset_new.csv')
exclude = set(meta['address']) | set(dx['address'].str.lower())
cand = cand[~cand['address'].isin(exclude)].reset_index(drop=True)
print('kandidat setelah membuang alamat rug pull dan yang sudah dipakai:', len(cand))

CSPAN = f'{BASE}/features/cand_span.csv'
done = set(pd.read_csv(CSPAN)['address']) if os.path.exists(CSPAN) else set()
T0, T1 = pd.Timestamp('2020-01-01').timestamp(), pd.Timestamp('2024-01-01').timestamp()

def tx_ts(addr, sort):
    for _ in range(5):
        try:
            j = requests.get('https://api.etherscan.io/v2/api', timeout=30, params=dict(chainid=1, module='account',
                  action='tokentx', contractaddress=addr, page=1, offset=1, sort=sort, apikey=KEY)).json()
            res = j.get('result')
            if isinstance(res, list): return int(res[0]['timeStamp']) if res else np.nan
            if isinstance(res, str) and 'rate limit' in res.lower(): time.sleep(2); continue
            return np.nan
        except Exception:
            time.sleep(2)
    return -1                         # gagal; JANGAN dianggap 'tidak ada transfer'

for i, a in enumerate(cand['address']):
    if a in done: continue
    first = tx_ts(a, 'asc'); time.sleep(0.22)
    if first == -1: continue
    last = np.nan
    if first == first and T0 <= first < T1:           # hanya yang peluncurannya 2020-2023
        last = tx_ts(a, 'desc'); time.sleep(0.22)
        if last == -1: continue
    pd.DataFrame([dict(address=a, first_ts=first, last_ts=last)]).to_csv(CSPAN, mode='a', header=not os.path.exists(CSPAN), index=False)
    if i % 300 == 0: print(i, '/', len(cand))

cs = pd.read_csv(CSPAN).drop_duplicates('address')
cs['year'] = pd.to_datetime(cs['first_ts'], unit='s').dt.year
cs['life_days'] = (cs['last_ts'] - cs['first_ts'])/86400
print(cs['year'].value_counts().sort_index())
print('2020-2023 & hidup >= 365 hari:'); print(cs[cs['life_days'] >= 365]['year'].value_counts().sort_index())

In [ ]:
cs = pd.read_csv(f'{BASE}/features/cand_span.csv').drop_duplicates('address')
cs['year'] = pd.to_datetime(cs['first_ts'], unit='s').dt.year
cs['life_days'] = (cs['last_ts'] - cs['first_ts']) / 86400
pool = cs[cs['year'].between(2020, 2023) & (cs['life_days'] >= 365)]
sel = pd.concat([pool[pool.year == 2021], pool[pool.year == 2022],
                 pool[pool.year == 2020].sample(25, random_state=0),
                 pool[pool.year == 2023].sample(25, random_state=0)])
print(sel['year'].value_counts().sort_index(), '| total diproses:', len(sel))

COLS = ['address','label','status'] + FEAT + ['slither_status','source_kind','is_proxy','n_files','erc20_like','solc_minor']
def build_one(a):
    res = fetch(a)
    if not res: return dict(address=a, status='fetch_failed')
    is_proxy = 0
    if res.get('Proxy') == '1' and res.get('Implementation'):
        r2 = fetch(res['Implementation'].lower())
        if r2 and r2.get('SourceCode'): res, is_proxy = r2, 1
    u = unpack(res)
    if not u: return dict(address=a, status='no_verified_source')
    files, remaps = u
    source = '\n'.join(files.values()); d = write_tree(a, files)
    det, st = run_slither(d, pick_main(files, res.get('ContractName')), solc_ver(res), remaps)
    rec = dict(address=a, label=0, status='built')
    rec.update(slither_feats(det) if det is not None else {c: np.nan for c in SL_COLS})
    rec.update(text_feats(source))
    rec.update(slither_status=st, source_kind='multi_file' if len(files) > 1 else 'single', is_proxy=is_proxy,
               n_files=len(files), erc20_like=int(E20 <= abi_names(res)), solc_minor=solc_minor(res))
    return rec

# UJI DULU pada 5 alamat
for a in sel['address'].head(5):
    r = build_one(a); print(a, {k: r.get(k) for k in ('status','slither_status','source_kind','is_proxy','erc20_like','solc_minor')})

In [ ]:
NEW = f'{BASE}/features/feature_v2_legit_new.csv'
done = set(pd.read_csv(NEW)['address']) if os.path.exists(NEW) else set()
for k, a in enumerate(sel['address']):
    if a in done: continue
    try: rec = build_one(a)
    except Exception as e: rec = dict(address=a, status=f'error:{type(e).__name__}')
    pd.DataFrame([rec]).reindex(columns=COLS).to_csv(NEW, mode='a', header=not os.path.exists(NEW), index=False)
    if k % 50 == 0: print(k, '/', len(sel))

new = pd.read_csv(NEW)
print(new['status'].value_counts(dropna=False))
ok = (new[(new.status == 'built') & (new.slither_status == 'ok') & (new.erc20_like == 1)]
      .merge(sel[['address','year','life_days','first_ts','last_ts']], on='address'))
print('funnel:', len(sel), '->', (new.status == 'built').sum(), 'terverifikasi ->',
      ((new.status == 'built') & (new.slither_status == 'ok')).sum(), 'Slither ok ->', len(ok), 'ERC-20')
print(ok['year'].value_counts().sort_index()); print(ok['source_kind'].value_counts())

In [ ]:
Rg = main[(main.label == 1) & (main.erc20_like == 1) & main['year'].between(2020, 2023)].copy()
p = Rg['year'].value_counts(normalize=True); avail = ok['year'].value_counts()
N = int(min(avail.get(y, 0) / p[y] for y in p.index))
Lg = pd.concat([ok[ok.year == y].sample(int(round(p[y] * N)), random_state=0) for y in p.index if int(round(p[y] * N)) > 0])
Lg['label'] = 0
keep = ['address','label','source_kind','is_proxy','year','solc_minor'] + FEAT
Dmain = pd.concat([Rg[keep], Lg[keep]]).reset_index(drop=True)
Dmain.to_csv(f'{BASE}/features/D_main_era_matched.csv', index=False)
print(Dmain['label'].value_counts())
print(pd.crosstab(Dmain['label'], Dmain['year'], normalize='index').round(3))
print(pd.crosstab(Dmain['label'], Dmain['solc_minor'], normalize='index').round(3))

In [ ]:
keep = ['address','label','source_kind','is_proxy','year','solc_minor'] + FEAT
rug_s = main[(main.label==1) & (main.erc20_like==1) & main['year'].between(2020, 2023) & (main.source_kind=='single')]
ok_s  = ok[ok.source_kind=='single']
p = rug_s['year'].value_counts(normalize=True)
N = int(min(ok_s['year'].value_counts().get(y, 0) / p[y] for y in p.index))
leg_s = pd.concat([ok_s[ok_s.year==y].sample(int(round(p[y]*N)), random_state=0)
                   for y in p.index if int(round(p[y]*N)) > 0]).assign(label=0)
D_s = pd.concat([rug_s[keep], leg_s[keep]]).reset_index(drop=True)
D_s.to_csv(f'{BASE}/features/D_single_era_matched.csv', index=False)
print(D_s['label'].value_counts())
print(pd.crosstab(D_s['label'], D_s['year'], normalize='index').round(3))
print(pd.crosstab(D_s['label'], D_s['solc_minor'], normalize='index').round(3))
print(pd.crosstab(D_s['label'], D_s['is_proxy']))

In [ ]:
META = ['year', 'solc_minor']
report(cv_eval(D_s, META, only='XGBoost'), name='Metadata saja (tahun + solc)')
report(cv_eval(D_s, FEAT), name='D_single: 29 fitur kode')
report(cv_eval(D_s, FEAT + META, only='XGBoost'), name='D_single: 29 fitur + metadata')

def holm(p):
    p = np.array(p, float); o = np.argsort(p); adj = np.empty(len(p)); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
    return adj

fx, ox = cv_eval2(D_s, FEAT)
fm, _  = cv_eval2(D_s, META, only='XGBoost')
Xf = fx[fx.model == 'XGBoost']
rows = []
for name, other in [('XGBoost vs RandomForest', fx[fx.model=='RandomForest']),
                    ('XGBoost vs LogReg', fx[fx.model=='LogReg']),
                    ('XGBoost(kode) vs metadata saja', fm)]:
    for met in ['mcc', 'pr_auc_legit']:
        d, pv = paired_nb(Xf, other, met); rows.append((name, met, d, pv))
for (n, m, d, pv), a in zip(rows, holm([r[3] for r in rows])):
    print(f'{n:32s} {m:13s} selisih={d:+.3f}  p={pv:.4f}  p_Holm={a:.4f}')

In [ ]:
sl_sev=['total_vulnerabilities','high_severity','medium_severity','low_severity','informational','optimization']
sl_flag=['has_assembly','has_low_level_calls','has_dead_code','has_shadowing','has_costly_loop','has_reentrancy']
rx=['has_mint','has_blacklist','has_pause','has_selfdestruct','has_owner_only','has_fee_modifier',
    'has_max_tx','has_whitelist','has_transfer_lock','has_proxy','has_renounce','has_burn']
st=['func_count','modifier_count','event_count','require_count','code_length']
abl = {'tanpa Slither': rx+st, 'tanpa Regex': sl_sev+sl_flag+st, 'tanpa Struktural': sl_sev+sl_flag+rx,
       'Slither saja': sl_sev+sl_flag, 'Regex saja': rx, 'Struktural saja': st}
rows = []
for name, cols in abl.items():
    f_, _ = cv_eval2(D_s, cols, only='XGBoost')
    d, pv = paired_nb(Xf, f_, 'mcc'); rows.append((name, f_.mcc.mean(), f_.pr_auc_legit.mean(), d, pv))
print(f'Semua 29: MCC={Xf.mcc.mean():.3f}  PR-AUC legit={Xf.pr_auc_legit.mean():.3f}')
for (n, mc, pa, d, pv), a in zip(rows, holm([r[4] for r in rows])):
    print(f'{n:18s} MCC={mc:.3f} PR-AUC={pa:.3f} | Semua29-ini: {d:+.3f}  p={pv:.4f}  p_Holm={a:.4f}')

In [ ]:
Dm = pd.read_csv(f'{BASE}/features/D_main_era_matched.csv'); Dm['multi'] = (Dm['source_kind']=='multi_file').astype(int)
report(cv_eval(Dm, META + ['multi'], only='XGBoost'), name='D_main: metadata + multi-file saja')
report(cv_eval(Dm, FEAT, only='XGBoost'), name='D_main: 29 fitur kode')

In [ ]:
def make_groups(d, mode='default'):
    key = [c for c in FEAT if c != 'code_length']
    sig = d[key].astype(int).astype(str).agg('|'.join, axis=1)
    if mode == 'default': sig = sig + '|' + (d['code_length'] // 1000).astype(int).astype(str)
    return sig.factorize()[0]

def cv_eval_g(d, cols, mode='default', n_repeats=5, n_splits=5, only=None):
    d = d.reset_index(drop=True); X = d[cols]; y = d['label'].values; g = make_groups(d, mode); rows = []
    for rep in range(n_repeats):
        for tr, te in StratifiedGroupKFold(n_splits, shuffle=True, random_state=rep).split(X, y, g):
            if len(set(y[te])) < 2: continue
            for name, m in make_models(rep).items():
                if only and name != only: continue
                if name == 'XGBoost': m.fit(X.iloc[tr], y[tr], sample_weight=compute_sample_weight('balanced', y[tr]))
                else: m.fit(X.iloc[tr], y[tr])
                p = m.predict_proba(X.iloc[te])[:, 1]; pr = (p >= 0.5).astype(int)
                tn, fp, fn, tp = confusion_matrix(y[te], pr, labels=[0, 1]).ravel()
                rows.append(dict(model=name, mcc=matthews_corrcoef(y[te], pr), bal_acc=balanced_accuracy_score(y[te], pr),
                    spec=tn/max(tn+fp, 1), rec=tp/max(tp+fn, 1), pr_auc_legit=average_precision_score(1-y[te], 1-p),
                    roc_auc=roc_auc_score(y[te], p), tn=tn, fp=fp, fn=fn, tp=tp, n_legit=(y[te] == 0).sum()))
    return pd.DataFrame(rows)

print('kluster longgar:', len(set(make_groups(D_s, 'default'))), '| kluster ketat:', len(set(make_groups(D_s, 'strict'))))
report(cv_eval_g(D_s, FEAT, mode='strict', only='XGBoost'), name='1. Grup ketat (abaikan code_length)')

key = [c for c in FEAT if c != 'code_length']
sig = D_s[key].astype(int).astype(str).agg('|'.join, axis=1)
D_u = D_s.loc[~sig.duplicated()].reset_index(drop=True)
print(D_u['label'].value_counts().to_dict())
report(cv_eval_g(D_u, FEAT, mode='strict', only='XGBoost'), name='2. Satu kontrak per kluster')

report(cv_eval_g(D_s[D_s.is_proxy == 0], FEAT, only='XGBoost'), name='3. Tanpa proxy')
for v in (6, 8):
    S = D_s[D_s.solc_minor == v]; print(f'solc 0.{v}.x:', S['label'].value_counts().to_dict())
    report(cv_eval_g(S, FEAT, only='XGBoost'), name=f'4. solc 0.{v}.x saja')

# 5. Uji temporal: latih 2020-2021, uji 2022-2023
tr, te = D_s[D_s.year <= 2021], D_s[D_s.year >= 2022]
m = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                      eval_metric='logloss', random_state=0)
m.fit(tr[FEAT], tr['label'], sample_weight=compute_sample_weight('balanced', tr['label']))
p = m.predict_proba(te[FEAT])[:, 1]; pr = (p >= 0.5).astype(int); yt = te['label'].values
tn, fp, fn, tp = confusion_matrix(yt, pr, labels=[0, 1]).ravel()
print('\n5. Temporal: latih', len(tr), '| uji', len(te), '| legit uji:', (yt == 0).sum(),
      '| MCC %.3f | spes %.3f | recall %.3f | PR-AUC legit %.3f | ROC-AUC %.3f' % (matthews_corrcoef(yt, pr),
      tn/(tn+fp), tp/(tp+fn), average_precision_score(1-yt, 1-p), roc_auc_score(yt, p)))

In [ ]:
# --- prediksi out-of-fold + analisis error ---
fx2, ox2 = cv_eval2(D_s, FEAT, only='XGBoost')
d_ = D_s.reset_index(drop=True)
q = ox2.groupby('i').agg(p=('p', 'mean'))
d_['p_rug'] = q['p'].reindex(range(len(d_))).values; d_['pred'] = (d_['p_rug'] >= 0.5).astype(int)
d_ = d_.merge(sel[['address', 'life_days']], on='address', how='left')
os.makedirs(f'{BASE}/outputs/metrics', exist_ok=True); d_.to_csv(f'{BASE}/outputs/metrics/oof_xgb_final.csv', index=False)

leg, rug = d_[d_.label == 0].copy(), d_[d_.label == 1]
print('Legit salah ditandai, per solc:');  print(leg.groupby('solc_minor')['pred'].agg(['mean', 'count']).round(3))
print('Legit salah ditandai, per proxy:'); print(leg.groupby('is_proxy')['pred'].agg(['mean', 'count']).round(3))
leg['masa_hidup'] = pd.qcut(leg['life_days'], 3, labels=['pendek', 'sedang', 'panjang'])
print('Legit salah ditandai, per masa hidup:'); print(leg.groupby('masa_hidup', observed=True)['pred'].agg(['mean', 'count']).round(3))
print('Rug terlewat, per solc:'); print(rug.groupby('solc_minor')['pred'].agg(lambda s: (s == 0).mean()).round(3))

# --- recall per tipe rug pull (data dianxiang) ---
dx = pd.read_csv(f'{BASE}/dataset/dianxiang/rugpull_full_dataset_new.csv'); dx['address'] = dx['address'].str.lower()
t = rug.merge(dx[['address', 'Type']].drop_duplicates('address'), on='address', how='left'); t['Type'] = t['Type'].fillna('(tidak ada di dianxiang)')
print(t.groupby('Type')['pred'].agg(n='count', recall=lambda s: (s == 1).mean()).sort_values('n', ascending=False).round(3))

# --- titik operasi (deskriptif) ---
fpr, tpr, thr = roc_curve(d_['label'], d_['p_rug'])
for target in [0.90, 0.95, 0.99]:
    j = np.argmax(tpr >= target); print(f'recall rug >= {target:.2f} -> spesifisitas {1-fpr[j]:.3f} (threshold {thr[j]:.3f})')

# --- SHAP asli, out-of-fold ---
import shap, matplotlib.pyplot as plt
os.makedirs(f'{BASE}/outputs/shap_plots', exist_ok=True)
d = D_s.reset_index(drop=True); X = d[FEAT]; y = d['label'].values; g = make_groups(d, 'default')
sv = np.zeros((len(d), len(FEAT))); base = np.zeros(len(d))
for tr_i, te_i in StratifiedGroupKFold(5, shuffle=True, random_state=0).split(X, y, g):
    m = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                          eval_metric='logloss', random_state=0)
    m.fit(X.iloc[tr_i], y[tr_i], sample_weight=compute_sample_weight('balanced', y[tr_i]))
    e = shap.TreeExplainer(m)(X.iloc[te_i]); sv[te_i] = e.values; base[te_i] = e.base_values
imp = pd.DataFrame({'feature': FEAT, 'mean_abs_shap': np.abs(sv).mean(0),
      'arah_korelasi': [np.corrcoef(X[f], sv[:, i])[0, 1] if X[f].nunique() > 1 else np.nan for i, f in enumerate(FEAT)]
      }).sort_values('mean_abs_shap', ascending=False)
print(imp.head(15).round(3).to_string(index=False)); imp.to_csv(f'{BASE}/outputs/shap_plots/shap_importance_final.csv', index=False)
plt.figure(); shap.summary_plot(sv, X, show=False, max_display=15)
plt.savefig(f'{BASE}/outputs/shap_plots/shap_summary_final.png', dpi=300, bbox_inches='tight'); plt.close()
for f in imp['feature'].head(3):
    shap.dependence_plot(f, sv, X, interaction_index=None, show=False)
    plt.savefig(f'{BASE}/outputs/shap_plots/dep_{f}.png', dpi=300, bbox_inches='tight'); plt.close()
def wf(i, name):
    ex = shap.Explanation(values=sv[i], base_values=base[i], data=X.iloc[i].values, feature_names=FEAT)
    plt.figure(); shap.plots.waterfall(ex, show=False); plt.savefig(f'{BASE}/outputs/shap_plots/{name}.png', dpi=300, bbox_inches='tight'); plt.close()
wf(d_[(d_.label == 1) & (d_.pred == 1)].sample(1, random_state=1).index[0], 'waterfall_rug_final')
wf(d_[(d_.label == 0) & (d_.pred == 0)].sample(1, random_state=1).index[0], 'waterfall_legit_final')

In [ ]:
cg_map = cand.set_index('address')[['cg_id', 'symbol']]
fp = d_[(d_.label == 0) & (d_.pred == 1)].sort_values('p_rug', ascending=False).head(20).join(cg_map, on='address')
print(fp[['address', 'cg_id', 'symbol', 'year', 'life_days', 'solc_minor', 'p_rug']].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, confusion_matrix
FIG = f'{BASE}/outputs/figures'; os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
dd = D_s.reset_index(drop=True); yv = dd['label'].values

# (a) model dan baseline metadata
grp = {'Metadata only': fm[fm.model=='XGBoost'], 'LogReg': fx[fx.model=='LogReg'],
       'RandomForest': fx[fx.model=='RandomForest'], 'XGBoost': fx[fx.model=='XGBoost']}
fig, ax = plt.subplots(figsize=(6, 3.6)); w = 0.38; xs = np.arange(len(grp))
for k, (met, lab) in enumerate([('mcc', 'MCC'), ('pr_auc_legit', 'PR-AUC (legitimate class)')]):
    ax.bar(xs + (k-.5)*w, [v[met].mean() for v in grp.values()], w, yerr=[v[met].std() for v in grp.values()], capsize=3, label=lab)
ax.set_xticks(xs); ax.set_xticklabels(grp.keys(), rotation=15); ax.set_ylim(0, 1.05); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(f'{FIG}/fig_models.png', dpi=300); plt.close()

# (b) ablation
abl_all = {'All 29 features': FEAT, 'Without Slither': rx+st, 'Without regex': sl_sev+sl_flag+st,
           'Without structural': sl_sev+sl_flag+rx, 'Slither only': sl_sev+sl_flag, 'Regex only': rx, 'Structural only': st}
res = {k: cv_eval2(D_s, c, only='XGBoost')[0] for k, c in abl_all.items()}
ks = list(res); fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.barh(ks[::-1], [res[k].mcc.mean() for k in ks][::-1], xerr=[res[k].mcc.std() for k in ks][::-1], capsize=3)
ax.set_xlabel('MCC (mean ± SD across folds)'); plt.tight_layout(); plt.savefig(f'{FIG}/fig_ablation.png', dpi=300); plt.close()

# (c) kurva ROC dan PR (kelas legit), prediksi out-of-fold dirata-rata antar ulangan
fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
for name in ['XGBoost', 'RandomForest', 'LogReg']:
    q = ox[ox.model == name].groupby('i').agg(p=('p', 'mean')).sort_index(); y_ = yv[q.index]; p_ = q['p'].values
    f_, t_, _ = roc_curve(y_, p_); ax[0].plot(f_, t_, label=name)
    pr_, rc_, _ = precision_recall_curve(1-y_, 1-p_); ax[1].plot(rc_, pr_, label=name)
ax[0].plot([0, 1], [0, 1], 'k:', lw=.8); ax[0].set_xlabel('False positive rate'); ax[0].set_ylabel('True positive rate (rug pull)')
ax[1].set_xlabel('Recall (legitimate class)'); ax[1].set_ylabel('Precision (legitimate class)'); ax[0].legend(frameon=False)
plt.tight_layout(); plt.savefig(f'{FIG}/fig_curves.png', dpi=300); plt.close()

# (d) matriks konfusi (prediksi out-of-fold dirata-rata antar ulangan, threshold 0.5)
cm = confusion_matrix(d_['label'], d_['pred'], labels=[0, 1])
fig, ax = plt.subplots(figsize=(4.2, 3.6)); im = ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_xticks([0, 1]); ax.set_xticklabels(['Legitimate', 'Rug pull']); ax.set_yticks([0, 1]); ax.set_yticklabels(['Legitimate', 'Rug pull'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); plt.tight_layout(); plt.savefig(f'{FIG}/fig_confusion.png', dpi=300); plt.close()

In [ ]:
!pip freeze | grep -iE "xgboost|scikit-learn|shap|slither|pandas|numpy|imbalanced" > {BASE}/outputs/versions.txt

In [ ]:
chk = D_s.set_index('address')
cols = ['has_max_tx', 'has_fee_modifier', 'has_renounce', 'has_low_level_calls', 'has_blacklist', 'has_pause']
print(pd.DataFrame({'20 legit teratas ditandai': chk.loc[fp['address'], cols].mean(),
                    'semua legit': chk[chk.label == 0][cols].mean(),
                    'semua rug pull': chk[chk.label == 1][cols].mean()}).round(2))

In [ ]:
D_t = D_s.copy(); D_t['sig'] = make_groups(D_t, 'strict')
tr, te = D_t[D_t.year <= 2021], D_t[D_t.year >= 2022].copy()
m = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                      eval_metric='logloss', random_state=0)
m.fit(tr[FEAT], tr['label'], sample_weight=compute_sample_weight('balanced', tr['label']))
te['pred'] = (m.predict_proba(te[FEAT])[:, 1] >= 0.5).astype(int)
te['signature_pernah_dilihat'] = te['sig'].isin(set(tr['sig']))
print(te.groupby(['label', 'signature_pernah_dilihat'])['pred'].agg(n='count', frac_diprediksi_rug='mean').round(3))

In [ ]:
# Figure 2 (perbaikan legenda, label sumbu, garis acak)
fig, ax = plt.subplots(figsize=(6.4, 3.8)); w = 0.38; xs = np.arange(len(grp))
for k, (met, lab) in enumerate([('mcc', 'MCC'), ('pr_auc_legit', 'PR-AUC (legitimate class)')]):
    ax.bar(xs + (k-.5)*w, [v[met].mean() for v in grp.values()], w, yerr=[v[met].std() for v in grp.values()], capsize=3, label=lab)
ax.axhline(0.182, color='gray', ls=':', lw=1)
ax.set_xticks(xs); ax.set_xticklabels(['Metadata only', 'Logistic regression', 'Random forest', 'XGBoost'], rotation=15)
ax.set_ylabel('Score (mean ± SD across folds)'); ax.set_ylim(0, 1.2); ax.legend(frameon=False, ncol=2, loc='upper left')
plt.tight_layout(); plt.savefig(f'{FIG}/fig_models.png', dpi=300); plt.close()

# Figure 5 (hitungan rata-rata per ulangan, sama dengan teks)
r = cv_eval(D_s, FEAT, only='XGBoost'); c = (r[['tn', 'fp', 'fn', 'tp']].sum() / 5).round().astype(int)
cm = np.array([[c.tn, c.fp], [c.fn, c.tp]]); print(cm)          # harus 372, 42, 66, 1798
fig, ax = plt.subplots(figsize=(4.2, 3.6)); ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_xticks([0, 1]); ax.set_xticklabels(['Legitimate', 'Rug pull']); ax.set_yticks([0, 1]); ax.set_yticklabels(['Legitimate', 'Rug pull'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); plt.tight_layout(); plt.savefig(f'{FIG}/fig_confusion.png', dpi=300); plt.close()

In [ ]:
import sys, xgboost, shap, sklearn, pandas, numpy
print(sys.version.split()[0], '|', xgboost.__version__, shap.__version__, sklearn.__version__, pandas.__version__, numpy.__version__)

In [ ]:
print('BASE' in globals(), 'FEAT' in globals(), 'sel' in globals())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

sel diagnostik 01


In [ ]:
dx = pd.read_csv(f'{BASE}/dataset/dianxiang/rugpull_full_dataset_new.csv'); dx['address'] = dx['address'].astype(str).str.lower()
print(dx['Chain'].value_counts().to_dict())
dx_eth = set(dx.loc[dx['Chain'] == 'ETH', 'address'])
gt = pd.read_excel(f'{BASE}/dataset/crpwarner/groundtruth/groundTruth.xlsx'); print(list(gt.columns)[:4])
gt_addr = set(gt['address'].astype(str).str.lower())
rug = set(meta.loc[meta.label == 1, 'address'])
print('rug pull di meta         :', len(rug))
print('  ada di dianxiang (ETH) :', len(rug & dx_eth))
print('  ada di CRPWarner       :', len(rug & gt_addr))
print('  di kedua sumber        :', len(rug & dx_eth & gt_addr))
print('  di LUAR keduanya       :', len(rug - dx_eth - gt_addr), '<- harus 0 agar kalimat Method benar')
print('dianxiang ETH tidak ada di meta:', len(dx_eth - rug), '| CRPWarner tidak ada di meta:', len(gt_addr - rug))